# Introducing Hooks

**Hooks** let you run your own commands **before or after** Claude Code runs a tool — great for automated workflows: format code after an edit, run tests when files change, block access to certain files, lint and feed results back, log activity.


## How hooks fit the flow

Normal Claude Code flow:
```
your query + tool defs -> Claude -> (decides to use a tool) -> Claude Code executes it -> result
```

Hooks **insert into that flow**, running code **just before** or **just after** the tool executes.

Two common hook types (more in a later lesson):
- **PreToolUse** — runs *before* a tool is called
- **PostToolUse** — runs *after* a tool is called

## Where hooks are configured

In Claude settings files:

| File | Scope |
|------|-------|
| `~/.claude/settings.json` | **Global** — all projects |
| `.claude/settings.json` | **Project** — shared with the team |
| `.claude/settings.local.json` | **Project, not committed** — personal |

Write them by hand, or use the **`/hooks`** command inside Claude Code. The config has a section per hook event, each holding matchers + the commands to run.

## PreToolUse

Runs **before** a tool executes. A **`matcher`** targets which tool(s):

```json
"PreToolUse": [
  {
    "matcher": "Read",
    "hooks": [
      { "type": "command", "command": "node /home/hooks/read_hook.js" }
    ]
  }
]
```

Before the `Read` tool runs, this command fires. Your command **receives details about the tool call** Claude wants to make, and can:
- **allow** the operation to proceed, or
- **block** the tool call and send an error message back to Claude.

## PostToolUse

Runs **after** a tool executes:

```json
"PostToolUse": [
  {
    "matcher": "Write|Edit",
    "hooks": [
      { "type": "command", "command": "node /home/hooks/edit_hook.js" }
    ]
  }
]
```

Because the tool already ran, PostToolUse **can't block** it. But it can:
- run **follow-up operations** (e.g. format the file that was just edited), and
- **provide feedback** to Claude about the tool use.

(The `matcher` is a pattern — `Write|Edit` targets multiple tools.)

## Practical applications

- **Code formatting** — auto-format after edits
- **Testing** — run tests when files change
- **Access control** — block reading/editing specific files
- **Code quality** — run linters/type-checkers and feed results back
- **Logging** — track what Claude accesses/modifies
- **Validation** — enforce naming/coding standards

**Big picture:** PreToolUse hooks control *what Claude can do*; PostToolUse hooks *enhance what Claude has done* — a way to wire your own tools/processes into the loop.

## Currency note

Since the course, hooks expanded well beyond Pre/PostToolUse — other events include **UserPromptSubmit**, **Notification**, **Stop**, **SubagentStop**, **PreCompact**, and **SessionStart**. Mechanically: a hook command receives a **JSON payload on stdin** (tool name, inputs, cwd, etc.) and can influence behavior via its **exit code** (e.g. exit 2 = block) or by printing **JSON** (e.g. a `permissionDecision`, or feedback returned to Claude). The lesson's model is exactly right; there are just more events and richer control now.

Next: **Defining hooks** — the config structure in detail.